In [1]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
    

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by using a `while True` loop and checking whether the model returned any `function_call` items.

- On each iteration, it sends the full `messages` history to the model.
- If the model returns a function call, the code executes the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If the model returns only a normal message and no function calls, `has_function_calls` stays `False`.
- At the end of the iteration, the code does:

```python
if has_function_calls == False:
    break
```

So it stops only when the model answers without asking for any more tools.


In [4]:
from rag_helper import RAGBase

#making a subclass RAGTraced
class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            self.last_usage = {
                "input_tokens": usage.input_tokens,
                "output_tokens": usage.output_tokens,
            }

            return response
        

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [5]:
from starter import rag

traced_rag = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    instructions=rag.instructions,
    prompt_template=rag.prompt_template,
    model=rag.model,
)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest model response**.


In [7]:
traced_rag.rag(query)

'It keeps calling the model in a `while True` loop.\n\nAfter each model response, the code checks whether there were any `function_call` items:\n\n- if there are function calls, it runs the tools, appends the tool results to `messages`, and loops again\n- if there are no function calls, it `break`s out of the loop\n\nSo the stop condition is פשוט: **no function calls in the latest response**.'